In [1]:
#python libraries
import pandas as pd
import numpy as np

In [6]:
# data cleaning
# load LASSO regression coefficients
file_path = "~/Downloads/Complete Tables.xlsx"
lasso_coefficients = pd.read_excel(file_path, sheet_name="REM coef")

# remove intercept since it is not a feature
lasso_coefficients = lasso_coefficients[lasso_coefficients["description"] != "Intercept"]

# standardize drug names
lasso_coefficients["drug"] = lasso_coefficients["antidep"].str.upper()

In [7]:
# (a) find the top 5 features increasing remission for Bupropion
bupropion_features = lasso_coefficients[lasso_coefficients["drug"] == "BUPROPION"]

top_remission_features = (
    bupropion_features
    .sort_values("coef", ascending=False)
    .head(5)
)

print("\nTop 5 features increasing remission for Bupropion:\n")
print(top_remission_features[["description", "code", "coef"]])


Top 5 features increasing remission for Bupropion:

                                          description      code      coef
42  MAJOR DEPRESSIVE DISORDER, RECURRENT EPISODE, ...     29635  2.138730
43  MAJOR DEPRESSIVE DISORDER, RECURRENT EPISODE, ...     29636  2.076863
44  MAJOR DEPRESSIVE DISORDER, RECURRENT EPISODE, ...     29630  2.021288
45                             Last AD: Same & Remiss  adrm_1SR  1.907853
47                             Last AD: Diff & Remiss  adrm_3DR  0.771543


In [8]:
# (b) find the bottom 5 features that decrease remission for Bupropion
lowest_remission_features = (
    bupropion_features
    .sort_values("coef", ascending=True)
    .head(5)
)

print("\nBottom 5 features reducing remission for Bupropion:\n")
print(lowest_remission_features[["description", "code", "coef"]])


Bottom 5 features reducing remission for Bupropion:

                                description      code      coef
46                Last AD: Same & No Remiss  adrm_2SN -0.838516
48                Last AD: Diff & No Remiss  adrm_4DN -0.536125
51  OTHER AND UNSPECIFIED BIPOLAR DISORDERS     29689 -0.327240
53        NONDEPENDENT TOBACCO USE DISORDER      3051 -0.315443
55      OTHER PROBLEMS RELATED TO LIFESTYLE      V698 -0.230484


In [9]:
# (c) identify common ICD diagnosis families affecting both Bupropion and Citalopram
# extract the first three digits of ICD codes because the assignment specifies
# that ICD codes with the same first 3 digits should be treated as the same feature
lasso_coefficients["ICD3"] = (
    lasso_coefficients["code"]
    .astype(str)
    .str.extract(r'(\d{3})')
)

# keep only ICD9 diagnosis rows
icd_rows = lasso_coefficients[lasso_coefficients["ctype"] == "ICD9"]

# select rows for Bupropion and Citalopram
two_drug_icd = icd_rows[icd_rows["drug"].isin(["BUPROPION", "CITALOPRAM"])]

# group by ICD3 family and compute mean coefficient
shared_diagnosis_groups = (
    two_drug_icd
    .groupby(["ICD3", "drug"])["coef"]
    .mean()
    .unstack()
)

print("\nICD diagnosis families affecting both medications:\n")
print(shared_diagnosis_groups.dropna())


ICD diagnosis families affecting both medications:

drug  BUPROPION  CITALOPRAM
ICD3                       
296    1.036945    0.142576
300   -0.083452   -0.020829
305   -0.315443   -0.060774
309   -0.161731   -0.134253
367    0.151947    0.062300
723   -0.043859    0.134186
780   -0.065369   -0.051477


In [10]:
# (d) determine the most informative questions to differentiate the two drugs
comparison_data = lasso_coefficients[
    lasso_coefficients["drug"].isin(["BUPROPION", "CITALOPRAM"])
]

# create a feature comparison table where rows represent clinical features
# and columns represent the two medications
feature_comparison = comparison_data.pivot_table(
    index="description",
    columns="drug",
    values="coef"
).fillna(0)

# entropy is a concept from information theory that measures uncertainty
# a high entropy value means the information is more mixed or uncertain
# a low entropy value means the information is more decisive or informative
# here we use entropy to measure how differently a feature behaves
# between Bupropion and Citalopram based on the LASSO coefficients
# features with lower entropy show stronger differences between drugs
# and therefore make better diagnostic questions for distinguishing treatments

def entropy(values):
    probabilities = np.abs(values) / np.sum(np.abs(values))
    probabilities = probabilities[probabilities > 0]
    return -np.sum(probabilities * np.log2(probabilities))

# compute entropy score for each feature
feature_scores = feature_comparison.apply(entropy, axis=1)

# select the 10 most informative questions
best_questions = feature_scores.sort_values().head(10)

print("\nTop 10 most informative questions to differentiate Bupropion vs Citalopram:\n")

for i, feature in enumerate(best_questions.index, 1):
    print(f"{i}. Does the patient have: {feature}?")


Top 10 most informative questions to differentiate Bupropion vs Citalopram:

1. Does the patient have: ABDOMINAL PAIN, EPIGASTRIC?
2. Does the patient have: OTHER AND UNSPECIFIED HYPERLIPIDEMIA?
3. Does the patient have: OTHER CONVULSIONS?
4. Does the patient have: OTHER GENERAL COUNSELING AND ADVICE FOR CONTRACEPTIVE MANAGEMENT?
5. Does the patient have: OTHER MALAISE AND FATIGUE?
6. Does the patient have: OTHER PROBLEMS RELATED TO LIFESTYLE?
7. Does the patient have: OTHER SCREENING MAMMOGRAM?
8. Does the patient have: OTHER SEBORRHEIC KERATOSIS?
9. Does the patient have: OTHER AND UNSPECIFIED ALCOHOL DEPENDENCE, UNSPECIFIED DRUNKENNESS?
10. Does the patient have: OTHER SPECIFIED NONINFLAMMATORY DISORDER OF VAGINA?


In [ ]:
#Summary:
#Entropy measures the uncertainty in information. In this analysis, entropy is calculated using the absolute LASSO coefficients 
#for each feature across the two medications. If a feature affects both drugs similarly, the entropy will be higher because 
#it does not help distinguish between them. If a feature affects the drugs very differently, the entropy will be lower,
#meaning the feature provides more useful information for deciding which medication is more appropriate. Therefore, 
#features with low entropy values are selected as the most informative questions to differentiate between Bupropion and Citalopram